<a href="https://colab.research.google.com/github/Fiarma/Data-Science-Projects/blob/fiarma/Consumer_Electronics_Sales_Forecasting_Forecasting_Sales_for_Fast_Moving_Consumer_Electronics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Import packages

In [ ]:
# Importation du module random pour générer des nombres aléatoires et effectuer des opérations aléatoires
import random

# Importation de NumPy, une bibliothèque pour le calcul scientifique, notamment les opérations sur les tableaux et matrices
import numpy as np

# Importation de Matplotlib pour la création de graphiques et de visualisations
import matplotlib.pyplot as plt

# Importation de Pandas pour la manipulation et l'analyse de données, notamment les DataFrames
import pandas as pd

# Importation de Seaborn pour créer des visualisations statistiques attractives et informatives
import seaborn as sns

# Importation des fonctions pour calculer les métriques d'évaluation des modèles, telles que l'erreur quadratique moyenne et l'erreur absolue moyenne
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Importation de la fonction pour diviser les données en ensembles d'entraînement et de test
from sklearn.model_selection import train_test_split

# Importation de la fonction adfuller pour réaliser le test de Dickey-Fuller augmenté, utilisé pour vérifier la stationnarité des séries temporelles
from statsmodels.tsa.stattools import adfuller

# Importation de la classe ARIMA pour modéliser et prédire les séries temporelles avec des composantes autorégressives et des moyennes mobiles
from statsmodels.tsa.arima.model import ARIMA

# Importation des fonctions pour tracer les fonctions d'autocorrélation (ACF) et d'autocorrélation partielle (PACF)
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Importation de la classe ExponentialSmoothing pour lisser les séries temporelles et tenir compte de la tendance et de la saisonnalité
from statsmodels.tsa.holtwinters import ExponentialSmoothing # Import from the correct module

# Importation de la fonction pour créer un graphique d'autocorrélation d'une série temporelle
from pandas.plotting import autocorrelation_plot

# Importation de la bibliothèque Prophet, développée par Facebook, pour la prévision des séries temporelles avec des tendances et des composantes saisonnières complexes
from prophet import Prophet

# Importation du module warnings pour gérer les avertissements
import warnings

# Commande pour ignorer tous les avertissements, ce qui permet de garder la sortie de l'exécution propre
warnings.filterwarnings('ignore')


###Data Collection

The dataset will be brought into Python with Pandas

In [ ]:
!pip install gdown

import gdown

#https://drive.google.com/file/d/1Gem-Q0oUt-r5LA5x7WZED5CFHLY71Vnc/view?usp=sharing

file_id = '1Gem-Q0oUt-r5LA5x7WZED5CFHLY71Vnc'
output_file = 'dataset.csv'

gdown.download(id = file_id, output = output_file, quiet=False)

df = pd.read_csv(output_file)

Downloading...
From: https://drive.google.com/uc?id=1Gem-Q0oUt-r5LA5x7WZED5CFHLY71Vnc
To: /content/dataset.csv
100%|██████████| 16.7M/16.7M [00:00<00:00, 27.0MB/s]


###Seeing the first few rows of the data

In [ ]:
df.head()

,Product_ID,Category,Price,Date,Season,Market_Trend_Index,Competitor_Activity_Score,Consumer_Confidence_Index,Product_Specification_1,Product_Specification_2,Sales_Volume
0,1103,Laptop,105.32,2009-01-01,Winter,-1.859160,0.546694,84.680465,Spec_C,Long-Battery-Life,49
1,1436,Tablet,145.55,2009-01-01,Winter,-0.345587,0.940428,42.919288,Spec_C,Lightweight,69
2,1271,Smartphone,97.82,2009-01-01,Winter,-0.384738,0.751155,55.191268,Spec_B,Lightweight,50
3,1107,Laptop,64.00,2009-01-01,Winter,0.716763,0.125939,88.746454,Spec_B,High-Resolution,28
4,1072,Tablet,67.83,2009-01-01,Winter,-0.242074,-0.412932,67.947536,Spec_A,Long-Battery-Life,81


###Data preprocessing

We are going to be:

    -    Checking for missing values

    -   Removing duplicates rows

    -   Converting `Date` column to datetime format

**Check missing values**

In [ ]:
missing_values = df.isnull().sum()

missing_values

Product_ID                   0
Category                     0
Price                        0
Date                         0
Season                       0
Market_Trend_Index           0
Competitor_Activity_Score    0
Consumer_Confidence_Index    0
Product_Specification_1      0
Product_Specification_2      0
Sales_Volume                 0
dtype: int64

**Check for duplicates in the dataset and remove them if they are.**

In [ ]:
df.duplicated().any()

False

**Convrt `Date` column to datetime format**

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])

###Exploratory Data Analysis ( EDA )

We aim to look for :
   
   - Patterns,

   - Seasonality

In [ ]:
sales_by_date_category = df.groupby(['Date', 'Category']).sum()['Sales_Volume'].reset_index()

sales_by_date_category.head(10)

,Date,Category,Sales_Volume
0,2009-01-01,Accessories,619
1,2009-01-01,Laptop,790
2,2009-01-01,Smartphone,564
3,2009-01-01,Tablet,768
4,2009-01-02,Accessories,227
5,2009-01-02,Laptop,210
6,2009-01-02,Smartphone,99
7,2009-01-02,Tablet,419
8,2009-01-03,Accessories,230
9,2009-01-03,Laptop,31


*Plot the daily sales trend for each product category *

In [ ]:
# Get all the unique categories from the dataset
